In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Masking, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Subtract, Multiply, concatenate
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

DATASET_SLUG  = "siamese-data"   # <-- GANTI sesuai nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'aug_' if USE_AUGMENTED else ''
meta_f = 'aug_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# Data sudah dimuat pada sel di atas (hindari menjalankan load dua kali).
pass

In [ ]:
# ── Hyperparameter ────────────────────────────────────────────────────────────
BILSTM_UNITS = 128
DROPOUT      = 0.3
EPOCHS       = 100
BATCH_SIZE   = 32
PATIENCE     = 10

# Mode pelatihan:
#   'per_idpsj' — satu model per soal (IDPSJ); butuh cukup sampel per prompt.
#   'pooled'    — satu model untuk SEMUA soal; disarankan jika per-IDPSJ terlalu sedikit.
TRAIN_MODE = 'pooled'   # 'per_idpsj' | 'pooled'

# Split train/val/test di dalam satu IDPSJ (hanya dipakai jika TRAIN_MODE == 'per_idpsj')
PER_PSJ_TEST_FRAC     = 0.20
PER_PSJ_VAL_FRAC      = 0.25
PER_PSJ_RANDOM_STATE  = 42

# Split saat gabungan semua prompt (hanya dipakai jika TRAIN_MODE == 'pooled')
GLOBAL_TEST_FRAC     = 0.20
GLOBAL_VAL_FRAC      = 0.25
GLOBAL_RANDOM_STATE  = 42


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                bilstm_units=128, dropout=0.3):
    """
    Siamese BiLSTM with direct regression output (grade 1-10).

    Arsitektur:
      - bilstm_question : encoder khusus untuk pertanyaan
      - shared_bilstm   : encoder Siamese untuk answerkey & answer
    Fitur gabungan: [eq, ea, eak, |eak-ea|, eak⊙ea]  →  5×256D = 1280D
    Output: Dense(1, linear) → prediksi grade secara langsung
    """
    bilstm_q      = Bidirectional(
        LSTM(bilstm_units, return_sequences=False), name='bilstm_question'
    )
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=False), name='shared_bilstm'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    eq  = bilstm_q(Masking(mask_value=0.0)(inp_q))
    eak = shared_bilstm(Masking(mask_value=0.0)(inp_ak))
    ea  = shared_bilstm(Masking(mask_value=0.0)(inp_a))

    diff     = Subtract(name='diff')([eak, ea])
    abs_diff = Lambda(lambda x: tf.abs(x), name='abs_diff')(diff)
    had_prod = Multiply(name='had_prod')([eak, ea])

    merged = concatenate([eq, ea, eak, abs_diff, had_prod], name='merged')

    x   = Dense(256, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64,  activation='relu')(x)
    out = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out,
                  name='siamese_bilstm_direct')

    # Error <= delta diperlakukan seperti MSE, error > delta seperti MAE
    huber_loss = Huber(delta=1.0)
    model.compile(optimizer='adam', loss=huber_loss, metrics=['mae', 'mse'])
    return model


# Preview
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2]
)
_tmp.summary()
del _tmp

In [ ]:
# ── Pelatihan ─────────────────────────────────────────────────────────────────
# Sintetis (IDJwb diawali 'syn_') hanya train; val & test = data asli.
#   pooled    : satu model, semua IDPSJ (split acak pada seluruh data asli).
#   per_idpsj : satu model tiap IDPSJ (split hanya dalam prompt itu).

y_all   = metadata['grade'].values.astype(np.float32)
is_real = ~metadata['IDJwb'].astype(str).str.startswith('syn_')

fold_results = []

def get_split(arr, idx):
    return arr[idx].astype(np.float32)


def train_one_model(train_idx, val_idx, test_idx, tag, model_basename):
    """Latih satu model; simpan ke OUT_DIR; kembalikan dict hasil eval test."""
    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    X_q_tr  = get_split(questions_emb,  train_idx)
    X_ak_tr = get_split(answerkeys_emb, train_idx)
    X_a_tr  = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te  = get_split(questions_emb,  test_idx)
    X_ak_te = get_split(answerkeys_emb, test_idx)
    X_a_te  = get_split(answers_emb,    test_idx)

    model = build_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        bilstm_units = BILSTM_UNITS,
        dropout      = DROPOUT,
    )

    grade_int   = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map    = dict(zip(unique_g, counts_g))
    n_kelas     = len(unique_g)
    raw_w       = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w    = raw_w / raw_w.mean()
    print(f"  Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}  "
          f"mean={sample_w.mean():.2f}")

    reduce_lr = ReduceLROnPlateau(
        monitor='val_mae', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_mae', patience=PATIENCE,
        restore_best_weights=True, verbose=1
    )

    print("  Memulai proses training...")
    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[reduce_lr, early_stop], verbose=1
    )

    y_pred_raw = model.predict([X_q_te, X_ak_te, X_a_te], verbose=0).flatten()
    y_pred     = np.clip(np.round(y_pred_raw), 1, 10)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}")

    model_path = os.path.join(OUT_DIR, model_basename)
    model.save(model_path)
    print(f"  Model saved -> {model_path}")
    tf.keras.backend.clear_session()

    return {
        'run'    : len(fold_results) + 1,
        'idpsj'  : tag,
        'mode'   : TRAIN_MODE,
        'n_train': len(y_train),
        'n_val'  : len(y_val),
        'n_test' : len(y_test),
        'rmse'   : rmse,
        'mae'    : mae,
        'y_test' : y_test,
        'y_pred' : y_pred,
    }


# ── Pooled: satu model, semua prompt ─────────────────────────────────────────
if TRAIN_MODE == 'pooled':
    print("\n" + "=" * 60)
    print("Mode POOLED — satu model untuk semua IDPSJ")
    print("=" * 60)

    real_idx = metadata.index[is_real].values
    syn_idx  = metadata.index[~is_real].values

    if len(real_idx) < 3:
        raise ValueError(
            "Data asli kurang dari 3 baris; pooled tidak bisa di-split. "
            "Tambah data atau gunakan skema lain."
        )

    tr_val, test_idx = train_test_split(
        real_idx,
        test_size=GLOBAL_TEST_FRAC,
        random_state=GLOBAL_RANDOM_STATE,
        shuffle=True,
    )
    if len(tr_val) < 2:
        raise ValueError("Terlalu sedikit sampel untuk validation setelah split test.")

    train_real_idx, val_idx = train_test_split(
        tr_val,
        test_size=GLOBAL_VAL_FRAC,
        random_state=GLOBAL_RANDOM_STATE,
        shuffle=True,
    )
    train_idx = np.concatenate([train_real_idx, syn_idx])
    print(f"  Sintetis hanya di train: {len(syn_idx)}")

    fold_results.append(
        train_one_model(
            train_idx, val_idx, test_idx,
            tag='(semua IDPSJ)',
            model_basename='model_pooled.keras',
        )
    )
    print("\nSelesai: 1 model (pooled).")

# ── Per IDPSJ: tanpa cross-prompt antar soal ─────────────────────────────────
elif TRAIN_MODE == 'per_idpsj':
    idpsj_list = sorted(metadata['IDPSJ'].unique())
    n_parts    = len(idpsj_list)

for i, psj_id in enumerate(idpsj_list):
    print(f"\n{'='*60}")
    print(f"IDPSJ {i+1:02d}/{n_parts}  |  prompt = {psj_id}")

    real_idx = metadata.index[
        (metadata['IDPSJ'] == psj_id) & is_real
    ].values
    syn_idx = metadata.index[
        (metadata['IDPSJ'] == psj_id) & ~is_real
    ].values

    if len(real_idx) < 3:
        print(f"  Lewati: kurang dari 3 sampel asli (n={len(real_idx)})")
        continue

    rs = PER_PSJ_RANDOM_STATE + i
    tr_val, test_idx = train_test_split(
        real_idx,
        test_size=PER_PSJ_TEST_FRAC,
        random_state=rs,
        shuffle=True,
    )
    if len(tr_val) < 2:
        print(f"  Lewati: terlalu sedikit sampel untuk val setelah split test")
        continue

    train_real_idx, val_idx = train_test_split(
        tr_val,
        test_size=PER_PSJ_VAL_FRAC,
        random_state=rs,
        shuffle=True,
    )
    train_idx = np.concatenate([train_real_idx, syn_idx])

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}  "
          f"(sintetis di train: {len(syn_idx)})")

    X_q_tr  = get_split(questions_emb,  train_idx)
    X_ak_tr = get_split(answerkeys_emb, train_idx)
    X_a_tr  = get_split(answers_emb,    train_idx)

    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)

    X_q_te  = get_split(questions_emb,  test_idx)
    X_ak_te = get_split(answerkeys_emb, test_idx)
    X_a_te  = get_split(answers_emb,    test_idx)

    model = build_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        bilstm_units = BILSTM_UNITS,
        dropout      = DROPOUT
    )

    grade_int   = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map    = dict(zip(unique_g, counts_g))
    n_kelas     = len(unique_g)
    raw_w       = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w    = raw_w / raw_w.mean()
    print(f"  Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}  "
          f"mean={sample_w.mean():.2f}")

    reduce_lr = ReduceLROnPlateau(
        monitor='val_mae',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    early_stop = EarlyStopping(
        monitor='val_mae',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    )

    print("  Memulai proses training...")
    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[reduce_lr, early_stop], verbose=1
    )

    y_pred_raw = model.predict([X_q_te, X_ak_te, X_a_te], verbose=0).flatten()
    y_pred     = np.clip(np.round(y_pred_raw), 1, 10)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}")

    fold_results.append({
        'run'        : len(fold_results) + 1,
        'idpsj'      : psj_id,
        'n_train'    : len(y_train),
        'n_val'      : len(y_val),
        'n_test'     : len(y_test),
        'rmse'       : rmse,
        'mae'        : mae,
        'y_test'     : y_test,
        'y_pred'     : y_pred,
    })

    safe_id = str(psj_id).replace(os.sep, '_').replace('/', '_')
    model_path = os.path.join(OUT_DIR, f'model_idpsj_{i+1:02d}_{safe_id}.keras')
    model.save(model_path)
    print(f"  Model saved -> {model_path}")

    tf.keras.backend.clear_session()

print(f"\n\nSelesai: {len(fold_results)} model (per IDPSJ yang memenuhi syarat).")

In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

summary = pd.DataFrame([{
    'run'    : r['run'],
    'idpsj'  : r['idpsj'],
    'n_train': r['n_train'],
    'n_test' : r['n_test'],
    'RMSE'   : round(r['rmse'], 4),
    'MAE'    : round(r['mae'],  4),
} for r in fold_results])

print("=" * 60)
print("Hasil per IDPSJ (tanpa cross-prompt)")
print("=" * 60)
if len(summary) == 0:
    print("Tidak ada hasil (semua IDPSJ dilewati?).")
else:
    print(summary.to_string(index=False))
    print(f"\nRata-rata  RMSE : {summary['RMSE'].mean():.4f} ± {summary['RMSE'].std():.4f}")
    print(f"Rata-rata  MAE  : {summary['MAE'].mean():.4f}  ± {summary['MAE'].std():.4f}")

summary.to_csv(os.path.join(OUT_DIR, 'per_idpsj_results.csv'), index=False)

if len(fold_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ['RMSE', 'MAE']):
        ax.bar(summary['idpsj'].astype(str), summary[metric], color='steelblue', edgecolor='black')
        ax.axhline(summary[metric].mean(), color='red', linestyle='--', label=f'Mean {metric}')
        ax.set_title(f'{metric} per IDPSJ')
        ax.set_xlabel('IDPSJ')
        ax.set_ylabel(metric)
        ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'per_idpsj_metrics.png'), dpi=150)
    plt.show()

    y_all_true = np.concatenate([r['y_test'] for r in fold_results])
    y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])

    plt.figure(figsize=(6, 6))
    plt.scatter(y_all_true, y_all_pred, alpha=0.5, edgecolors='k', linewidths=0.3)
    plt.plot([1, 10], [1, 10], 'r--', label='Ideal')
    plt.xlabel('Grade Aktual')
    plt.ylabel('Grade Prediksi')
    plt.title('Prediksi vs Aktual (gabungan semua IDPSJ)')
    plt.xticks(range(1, 11))
    plt.yticks(range(1, 11))
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'scatter_all_idpsj.png'), dpi=150)
    plt.show()